# 2025년 서울 고령 보행자 교통사고 분석

**핵심 취재 질문:** 동대문구는 왜 고령 보행자 사고가 가장 많이 발생한 지역인가?

2025년 서울 25개 자치구의 사고 건수, 고령인구 대비 발생 수준, 전체 보행자 사고 중 고령 보행자 사고 비중을 차례로 비교한다. 수치로 확인된 사실과 아직 검증되지 않은 원인을 구분한다.

- 출처: 한국도로교통공단 TAAS(2025년 보행자·고령 보행자 사고), 서울 열린데이터광장(2025년 4분기 고령인구)
- 분석 도구: Python, pandas, Google Colab
- **해석 유의:** 거주 고령인구와 사고 발생 장소의 인구가 다르고, 실제 보행량도 알 수 없으므로 개인의 사고 위험률을 뜻하지 않는다.

## 1. 자료 불러오기

Colab에서 실행한다. `data/` 폴더의 세 파일을 아래 이름으로 준비해 한꺼번에 업로드한다. Colab 실행 환경은 로컬 VS Code 폴더와 자동으로 연결되지 않는다.

- `2025_고령보행자_사고.xlsx`
- `2025_고령인구.csv`
- `2025_전체보행자사고.xlsx`

In [ ]:
import pandas as pd
from google.colab import files

uploaded = files.upload()  # 세 파일을 선택

accident = pd.read_excel('2025_고령보행자_사고.xlsx')
all_pedestrian = pd.read_excel('2025_전체보행자사고.xlsx')

# 이 CSV는 위쪽 4행이 여러 단계의 열 제목이므로 열 이름 없이 읽는다.
population_raw = pd.read_csv('2025_고령인구.csv', skiprows=4, header=None)

print('고령 보행자 사고 원본:', accident.shape)
print('전체 보행자 사고 원본:', all_pedestrian.shape)
print('고령인구 원본:', population_raw.shape)

## 2. 전처리 — 서울 25개 자치구만 추출

`시도` 앞뒤의 공백을 제거하고 `사고[건]`만 선택한다. 인구 CSV에서는 두 번째 열(자치구), 여섯 번째 열(65세 이상 인구 소계)을 사용한다.

In [ ]:
def seoul_accident_counts(df, new_column):
    data = df.copy()
    data['시도'] = data['시도'].str.strip()
    data = data.loc[
        (data['시도'] == '서울') &
        (data['연도'] == '사고[건]') &
        (data['시군구'] != '합계'),
        ['시군구', '2025']
    ].copy()
    return data.rename(columns={'2025': new_column})

seoul_accident = seoul_accident_counts(accident, '사고건수')
seoul_all = seoul_accident_counts(all_pedestrian, '전체보행자사고')

population = population_raw.iloc[:, [1, 5]].copy()
population.columns = ['시군구', '고령인구']
population = population[population['시군구'] != '소계'].copy()
population['고령인구'] = pd.to_numeric(population['고령인구'])

print('고령 보행자 사고 지역 수:', len(seoul_accident))
print('전체 보행자 사고 지역 수:', len(seoul_all))
print('고령인구 지역 수:', len(population))
print('고령 보행자 사고 합계:', seoul_accident['사고건수'].sum())
print('전체 보행자 사고 합계:', seoul_all['전체보행자사고'].sum())

display(population.head())

## 3. 분석 1 — 고령인구 대비 사고 건수

**질문:** 거주 고령인구 규모를 고려해도 동대문구의 사고 발생 수준이 높은가?

**계산:** 고령인구 1만 명당 사고 건수 = 고령 보행자 사고 건수 ÷ 해당 자치구의 65세 이상 인구 × 10,000.

In [ ]:
merged = pd.merge(seoul_accident, population, on='시군구', how='inner', validate='one_to_one')
assert len(merged) == 25, '결합 후 지역 수가 25개인지 확인하세요.'

merged['1만명당_사고건수'] = merged['사고건수'] / merged['고령인구'] * 10000
merged['건수순위'] = merged['사고건수'].rank(ascending=False, method='min').astype(int)
merged['인구대비순위'] = merged['1만명당_사고건수'].rank(ascending=False, method='min').astype(int)

result = merged.sort_values('1만명당_사고건수', ascending=False)
display(result.round({'1만명당_사고건수': 2}))

### 분석 1에서 확인한 사실

- **동대문구:** 고령 보행자 사고 132건(서울 **1위**). 2025년 4분기 65세 이상 인구 72,779명 기준 **1만 명당 18.14건(2위)**.
- **중구:** 사고 61건(건수 18위), 고령인구 1만 명당 22.08건(1위).
- 동대문구의 사고 건수 1위는 단지 거주 고령인구가 많기 때문이라고 보기 어렵다. 다만 이 지표만으로 원인을 판단할 수 없다.

## 4. 분석 2 — 전체 보행자 사고 중 고령 보행자 사고 비중

**질문:** 동대문구에서 일어난 보행자 사고 가운데 고령 보행자 사고의 비중도 높은가?

**계산:** 고령 보행자 사고 비중(%) = 고령 보행자 사고 건수 ÷ 전체 보행자 사고 건수 × 100.

두 TAAS 통계가 모두 2025년 같은 지역에서 발생한 보행자 피해 사고를 집계한 자료라는 전제로 비교한다.

In [ ]:
comparison = pd.merge(merged, seoul_all, on='시군구', how='inner', validate='one_to_one')
assert len(comparison) == 25, '결합 후 지역 수가 25개인지 확인하세요.'
assert (comparison['사고건수'] <= comparison['전체보행자사고']).all()

comparison['고령사고비중'] = comparison['사고건수'] / comparison['전체보행자사고'] * 100
comparison['비중순위'] = comparison['고령사고비중'].rank(ascending=False, method='min').astype(int)
comparison_sorted = comparison.sort_values('고령사고비중', ascending=False)

display(comparison_sorted[
    ['시군구', '전체보행자사고', '사고건수', '고령사고비중', '비중순위']
].round({'고령사고비중': 2}))

seoul_share = seoul_accident['사고건수'].sum() / seoul_all['전체보행자사고'].sum() * 100
print(f'서울 전체 고령 보행자 사고 비중: {seoul_share:.2f}%')

### 분석 2에서 확인한 사실

- **동대문구:** 전체 보행자 사고 356건 중 고령 보행자 사고 132건, **37.08%(서울 3위)**.
- 서울 전체: 7,511건 중 2,060건, **27.43%**. 동대문구는 서울 전체보다 **약 9.65%p 높다**.
- **강북구 42.00%(1위)**, **은평구 37.62%(2위)**, 동대문구 37.08%(3위).
- 동대문구는 고령 보행자 사고 **건수 1위·고령인구 대비 2위·전체 사고 중 비중 3위**다.

**해석의 한계:** 고령사고 비중은 고령자 보행량뿐 아니라 다른 연령대의 사고 건수에도 좌우된다. 높은 비중을 곧바로 고령 보행자 개인의 더 높은 사고 위험으로 해석하지 않는다.

## 5. 앞으로 확인할 취재 질문

1. 동대문구의 고령 보행자 사고는 특정 교차로·도로·생활권에 집중되는가? **사고 위치 자료 확보 후 분석**.
2. 확인된 사고 집중 지점의 보행환경과 안전시설, 노인보호구역 지정·관리 상태는 어떠한가? **현장 취재 필요**.
3. 고령 보행자와 다른 연령대의 이동·보행량 차이가 지역별 지표에 어떤 영향을 주는가? **현재 자료로 판단 불가**.

분석에서 발견한 패턴은 취재를 시작할 근거이지, 사고 원인을 증명하는 결과가 아니다.